# 01. 말뭉치 개요 — LLM 에이전트 논문 전체와 한중일어 대상 집합 (2023-01 ~ 2026-09)

- **목적**: 토픽 학습 집합(세계 전체 = 수집한 다섯 cs 분류의 LLM 에이전트 논문)과 투영 집합(한국어·중국어·일본어 대상)의 규모·분포를 확인하고 논문 3장의 자료 표와 그림 1을 만든다. 방법 절에 적을 증빙(연도 기준, 중복 제거 편수)도 표로 남긴다.
- **입력**: `preprocessing/output/corpus_llm_agent_all_2023_2026q3.csv`, `corpus_llm_agent_cjk_lang_2023_2026q3.csv`, 정본 `verification/corpus_counts_*.json`(01_build_corpus.py)
- **정의** (DECISIONS)
  - LLM 에이전트 논문(D1): 제목·초록에 agent 계열 단어와 LLM·생성형 AI 계열 단어가 함께 나오는 논문
  - 대상 언어(D10·D36): 제목·초록 키워드로 잡은 korean / chinese / japanese 신호(257편)에 사람 검수 라벨을 붙였다. 집합에 넣는 것은 "그 언어의 텍스트를 실제 연구 대상으로 다룸"뿐이다. 영어와 그 언어만 쓴 논문은 실제 연구 대상, 영어·그 언어 외 다른 언어가 더 있으면 "다국어 실험의 한 언어로 포함"(집합 밖, 따로 센다). 라벨은 논문 단위라 복수 태그 논문은 모든 태그에 같은 라벨이다. 논문이 다루는 언어이며 쓰인 언어가 아니다
  - 연도(D31): arXiv id 접두의 최초 제출 월(YYMM). 수집본의 published는 최신 판 날짜라 쓰지 않는다. 기간은 2023-01~2026-09(D30), 2026은 9개월치
  - 소속 국가 축은 쓰지 않는다(D30). 2026-03~09 수집본에는 소속이 없다
- **주의**
  - 대상 언어 태그는 한 논문에 여러 개가 붙을 수 있어 언어 합이 집합 크기를 넘을 수 있다
  - 비율의 분모 "전체 cs 논문"은 다섯 분류(cs.AI/CL/CV/CY/LG)의 고유 논문 수다(정본 `pipeline.unique_by_year`)


In [ ]:
import glob, json, re, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "verification").exists())   # 노트북은 analysis/notebooks/ 에 있다
DATA = ROOT / "preprocessing/output"                                   # 입력: 전처리 산출(정제 CSV)
VER = ROOT / "verification"                                            # 수치의 정본 JSON
FIG = ROOT / "analysis/output/figures"; FIG.mkdir(parents=True, exist_ok=True)   # 출력: 노트북 그림. 원고 그림은 paper/figures/
TAB = ROOT / "analysis/output/tables"; TAB.mkdir(parents=True, exist_ok=True)    # 출력: 노트북 표
CACHE = ROOT / "analysis/output/cache"; CACHE.mkdir(parents=True, exist_ok=True) # 출력: 재실행용 캐시(git 제외)
TODAY = pd.Timestamp.today().strftime("%Y%m%d")
SUFFIX = "2023_2026q3"   # D30: 분석 기간 2023-01 ~ 2026-09 말뭉치

try:
    sys.path.insert(0, str(ROOT / "analysis/scripts"))
    import figure_style; figure_style.apply_style(); C = figure_style.COLORS; CYC = figure_style.COLOR_CYCLE
except Exception:
    C = {"green": "#46B07B", "orange": "#FF9E02", "teal": "#3A9E8F", "coral": "#E87B4A", "green_dark": "#2D7A53", "orange_dark": "#C47800", "green_light": "#8DD4AA", "orange_light": "#FFD080", "green_pale": "#C8EBDA", "orange_pale": "#FFECC0", "gray": "#7F8C8D", "gray_light": "#BDC3C7", "gray_dark": "#4A4A4A", "text": "#333333"}; CYC = [C["green"], C["orange"], C["teal"], C["coral"]]
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

LANGS = ["ko", "zh", "ja"]
LANG_NAME = {"ko": "Korean", "zh": "Chinese", "ja": "Japanese"}
YEARS = [2023, 2024, 2025, 2026]          # 2026은 1~9월(D30)
MIN_N = 10   # 투영 집합의 토픽별 건수가 이보다 작으면 lift를 보고하지 않는다

allc = pd.read_csv(DATA / f"corpus_llm_agent_all_{SUFFIX}.csv", dtype={"id": str})
cjk_signal = pd.read_csv(DATA / f"corpus_llm_agent_cjk_lang_{SUFFIX}.csv", dtype={"id": str})   # 키워드 신호 257편 + 사람 라벨(D36)
counts = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
YEAR_BASIS = counts["definitions"]["year_basis"]   # D31: "id" = arXiv id 접두(최초 제출 월)
LABEL_TARGET = "그 언어의 텍스트를 실제 연구 대상으로 다룸"; LABEL_MULTI = "다국어 실험의 한 언어로 포함"
cjk = cjk_signal[cjk_signal.cjk_lang].copy()       # 최종 대상 언어 집합 = 신호 ∧ 라벨 '실제 연구 대상'(D36)

# 정본 대조: 노트북이 읽은 자료가 정본 JSON과 같은지 확인한다
assert YEAR_BASIS == "id", "연도 기준이 D31(id 접두)이 아니다"
assert len(allc) == counts["pipeline"]["llm_agent_all"], "전체 건수가 정본과 다르다"
assert len(cjk_signal) == counts["pipeline"]["llm_agent_cjk_signal"], "대상 언어 신호 건수가 정본과 다르다"
assert len(cjk) == counts["pipeline"]["llm_agent_cjk_lang"], "대상 언어 집합 건수가 정본과 다르다"
assert (cjk_signal.lang_label != "").all(), "라벨 없는 신호 논문이 있다"
assert allc.groupby("year").size().to_dict() == {int(k): v for k, v in counts["llm_agent_by_year"].items()}
assert set(cjk.id) <= set(allc.id) and allc.id.is_unique
for l in LANGS:
    allc[f"lang_{l}"] = allc[f"lang_{l}"].astype(bool); allc[f"lang_{l}_signal"] = allc[f"lang_{l}_signal"].astype(bool)
assert {l: int(allc[f"lang_{l}"].sum()) for l in LANGS} == counts["cjk_lang_by_lang_total"]
N_ALL_BY_YEAR = {int(k): v for k, v in counts["pipeline"]["unique_by_year"].items()}   # 분모: 다섯 cs 분류의 고유 논문 수
print("정본 대조 통과: LLM 에이전트", len(allc), "| 한중일어 신호", len(cjk_signal), "| 대상 언어 집합(사람 라벨)", len(cjk), "| 연도 기준", YEAR_BASIS, "| 기간", counts["definitions"]["period"][:24])

## 1. 자료 표 — 연도별 전체 cs 논문·LLM 에이전트·한중일어 대상

- 계산: 연도별 다섯 분류 고유 논문 수, LLM 에이전트 논문 수와 비중(%), 그중 한국어·중국어·일본어 대상 논문 수. 합계 행을 둔다
- 출력: `analysis/output/tables/table1_year_overview.csv` (논문 3장 자료 표)


In [ ]:
t1 = pd.DataFrame({"cs papers (5 categories)": pd.Series(N_ALL_BY_YEAR), "LLM-agent papers": allc.groupby("year").size()}).reindex(YEARS).fillna(0).astype(int)
t1["LLM-agent share %"] = (100 * t1["LLM-agent papers"] / t1["cs papers (5 categories)"]).round(2)
for l in LANGS:
    t1[f"target {LANG_NAME[l]}"] = allc[allc[f"lang_{l}"]].groupby("year").size().reindex(YEARS).fillna(0).astype(int)
t1["target CJK (any)"] = cjk.groupby("year").size().reindex(YEARS).fillna(0).astype(int)
tot = t1.sum(numeric_only=True); tot["LLM-agent share %"] = round(100 * t1["LLM-agent papers"].sum() / t1["cs papers (5 categories)"].sum(), 2)
t1.loc["total"] = tot; t1.index = [str(i) if i != 2026 else "2026 (Jan-Sep)" for i in t1.index]
t1.to_csv(TAB / "table1_year_overview.csv"); t1

## 1b. 대상 언어 사람 검수 라벨 — 신호 257편의 라벨 분포 (D36)

- 계산: 키워드 신호(target_langs_signal)별 라벨 분포. 집합에 들어가는 것은 "그 언어의 텍스트를 실제 연구 대상으로 다룸"뿐. "다국어 실험의 한 언어로 포함"은 언어별로 따로 센다
- 입력: `cjk_signal`(257편, 라벨 열 lang_label), 정본 `language_label_review`(검수 파일 경로·재점검 변경 수)
- 출력: `analysis/output/tables/table1b_lang_label_review.csv` (방법 절의 검수 결과 표)
- 주의: 라벨은 논문 단위. 신호가 둘 이상인 논문(예 ko;ja)은 모든 태그에 같은 라벨. 에이전트 무관 분모(3절)는 검수하지 않았으므로 키워드 신호 그대로


In [ ]:
lr = counts["language_label_review"]
print(f"검수 파일: {lr['main_file']} ({lr['n_main']}편) + 재점검 {lr['recheck_file']} ({lr['n_recheck']}편, 라벨 바뀜 {lr['n_recheck_changed']}편)")
t1b = pd.crosstab(cjk_signal.target_langs_signal, cjk_signal.lang_label.replace("", "(없음)"), margins=True, margins_name="total")
assert int(t1b.loc["total", LABEL_TARGET]) == len(cjk)
t1b.to_csv(TAB / "table1b_lang_label_review.csv")
per_lang = pd.DataFrame({LANG_NAME[l]: {"signal": int(cjk_signal[f"lang_{l}_signal"].sum()), "target (in set)": int(cjk_signal[f"lang_{l}"].sum()),
                                        "multilingual (out)": int((cjk_signal[f"lang_{l}_signal"] & cjk_signal.lang_multilingual_human).sum()),
                                        "other (out)": int((cjk_signal[f"lang_{l}_signal"] & ~cjk_signal.lang_target_human & ~cjk_signal.lang_multilingual_human).sum())} for l in LANGS}).T
assert per_lang["target (in set)"].to_dict() == {LANG_NAME[l]: counts["cjk_lang_by_lang_total"][l] for l in LANGS}
print(per_lang.to_string()); t1b

## 1c. 민감도 — 라벨 규칙을 뒤집으면, 정의 밖에 남는 에이전트 논문은 (리뷰 2026-10-14)

- (가) 대상 언어 라벨 규칙 "영어와 그 언어만이면 실제 연구 대상"을 뒤집어 "영어가 언급되면 다국어 실험"으로 보면 집합이 얼마나 줄어드는가. 검수 입력(제목·초록)에서 'english'가 있는 실제-대상 논문을 센다
- (나) agent 계열 단어는 있고 LLM 계열 용어는 없어 정의(D1) 밖에 남은 논문 가운데 agentic, ai agent, coding/web/gui/computer-use agent, tool calling, mcp 표현이 있는 논문 수(연도별). 소문자 부분 문자열 일치로 센 어림값이며 정의를 바꾸지 않는다
- 출력: 정본 `sensitivity` 항목


In [ ]:
rv = cjk_signal.set_index("id")[["title", "abstract"]].to_dict(orient="index")
tgt = cjk_signal[cjk_signal.cjk_lang]
has_en = tgt.id.map(lambda i: "english" in (rv[i]["title"] + " " + rv[i]["abstract"]).lower())
rev_rule = {l: int((tgt[f"lang_{l}"] & ~has_en).sum()) for l in LANGS}; rev_rule["any"] = int((~has_en).sum())
print("규칙을 뒤집으면 남는 집합:", rev_rule, "| 현재", {l: int(tgt[f"lang_{l}"].sum()) for l in LANGS}, len(tgt))
tag = pd.read_csv(DATA / f"corpus_agent_tag_{SUFFIX}.csv", dtype={"id": str}, usecols=["id", "year", "title", "abstract", "is_llm_agent"])
out_def = tag[~tag.is_llm_agent.astype(bool)]; tx = (out_def.title.fillna("") + " " + out_def.abstract.fillna("")).str.lower()
CUES = ["agentic", "ai agent", "coding agent", "web agent", "gui agent", "computer-use agent", "computer use agent", "tool calling", "tool-calling", "function calling", " mcp"]
cue = tx.map(lambda x: any(c in x for c in CUES))
outside = out_def[cue].groupby("year").size().reindex(YEARS, fill_value=0)
print(f"agent 태그만 있고 정의 밖인 논문 {len(out_def):,}편 중 LLM 에이전트 단서 보유 {int(cue.sum()):,}편:", outside.to_dict())
sensitivity = {"label_rule_reversed_set": rev_rule, "label_rule_note": "'english'가 제목·초록에 있는 실제-대상 논문을 다국어로 간주했을 때 남는 편수",
               "agent_tag_outside_definition": int(len(out_def)), "outside_with_llm_agent_cues": {str(k): int(v) for k, v in outside.items()}, "cues": CUES}

## 2. 방법 절 증빙 — 연도 기준과 중복 제거

- 계산: (가) 같은 집합을 published(수집본 날짜) 기준으로 셌을 때의 연도별 고유 논문 수를 id 접두 기준과 나란히 둔다. (나) 제목이 같은 다른 id(재제출·중복 등록)를 한 편으로 합친 건수.
- 입력: 정본 `pipeline.unique_by_year`, `pipeline.unique_by_year_other_basis`, `pipeline.same_title_different_id`
- 출력: `analysis/output/tables/table0_year_basis_and_dedup.csv`


In [ ]:
pl = counts["pipeline"]
t0 = pd.DataFrame({"id prefix (first submission month, D31)": {int(k): v for k, v in pl["unique_by_year"].items()},
                   "published (harvest-time latest version)": {int(k): v for k, v in pl["unique_by_year_other_basis"].items()}}).sort_index().fillna(0).astype(int)
t0.to_csv(TAB / "table0_year_basis_and_dedup.csv")
dd = pl["same_title_different_id"]
print(f"같은 제목·다른 id: {dd['groups']}묶음 {dd['rows']}편 → {dd['removed']}편 제거 (규칙: {dd['rule']})")
print(f"교차 등재 포함 원 행 수 {pl['rows_raw_all_files']:,} → 고유 id {pl['unique_ids_all_years']:,} → 기간 안 {pl['unique_ids_in_period']:,}")
t0

## 3. 대상 언어 논문 안에서 에이전트의 비중

- 계산: 그 언어를 대상으로 한 논문 전체(에이전트 무관, 정본 `language_target_papers_all_topics`) 대비 LLM 에이전트 논문의 비율
- 해석: "한국어를 다루는 논문 중 에이전트 논문은 몇 %인가". 언어권 연구 안에서 에이전트 주제의 비중


In [ ]:
lt = counts["language_target_papers_all_topics"]
t3 = pd.DataFrame({LANG_NAME[l]: {"target-language papers (all topics)": lt[l]["total"], "LLM-agent among them": lt[l]["llm_agent"],
                                   "share %": round(100 * lt[l]["llm_agent"] / lt[l]["total"], 2)} for l in LANGS}).T
t3.to_csv(TAB / "table3_agent_share_within_language.csv"); t3

## 4. 그림 1 — 연도별 건수

- 왼쪽: 전체 LLM 에이전트 논문(막대)과 전체 cs 논문 대비 비중(선)
- 오른쪽: 대상 언어별 LLM 에이전트 논문 수(로그 축). 중국어와 한국어·일본어의 규모 차이를 한 축에 둔다
- 출력: `analysis/output/figures/fig1_year_counts.png`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.15]}); xl = [str(y) if y != 2026 else "2026\n(Jan–Sep)" for y in YEARS]
ax = axes[0]; tot = allc.groupby("year").size().reindex(YEARS)
ax.bar(range(4), tot.values, color=C["green"], width=0.62, label="LLM-agent papers", zorder=3)
for i, v in enumerate(tot.values): ax.text(i + 0.08, v + 0.02 * tot.max(), f"{v:,}", ha="left", va="bottom", fontsize=8, color=C["text"])
ax2 = ax.twinx(); share = 100 * tot / pd.Series(N_ALL_BY_YEAR).reindex(YEARS)
ax2.plot(range(4), share.values, marker="o", color=C["orange"], lw=1.8, label="share of cs papers (%)", zorder=4); ax2.set_ylim(0, max(14, share.max() + 1)); ax2.set_ylabel("share (%)"); ax2.grid(False); ax2.spines["top"].set_visible(False)
for i, v in enumerate(share.values): ax2.text(i - 0.1, v + 0.35, f"{v:.1f}%", fontsize=7.5, color=C["orange_dark"], ha="right", va="bottom")
ax.set_xticks(range(4)); ax.set_xticklabels(xl); ax.set_ylabel("papers"); ax.set_title("LLM-agent papers in five cs categories"); ax.set_ylim(0, tot.max() * 1.15)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="upper left")
ax = axes[1]; w = 0.26; cols = [C["orange"], C["green"], C["teal"]]
for i, l in enumerate(LANGS):
    vals = allc[allc[f"lang_{l}"]].groupby("year").size().reindex(YEARS).fillna(0).astype(int)
    bars = ax.bar(np.arange(4) + (i - 1) * w, vals.values, width=w, label=LANG_NAME[l], color=cols[i], zorder=3)
    for b, v in zip(bars, vals.values): ax.text(b.get_x() + b.get_width() / 2, max(v, 0.8) * 1.08, str(v), ha="center", va="bottom", fontsize=7, color=C["text"])
ax.set_yscale("log"); ax.set_ylim(0.7, 200); ax.set_xticks(range(4)); ax.set_xticklabels(xl); ax.set_ylabel("papers (log scale)"); ax.set_title("Target-language LLM-agent papers (human-labeled)"); ax.legend(loc="upper left")
fig.tight_layout(); fig.savefig(FIG / "fig1_year_counts.png", bbox_inches="tight", dpi=200); plt.show()

## 5. 정본 저장과 검수 목록

- 저장: 표 1·1b·0·3의 값을 `verification/overview_notebook_YYYYMMDD.json`에 남긴다(노트북 산출 정본). 한중일어 대상 논문 목록은 검수 입력 참고용 CSV


In [ ]:
rev = cjk[["id", "year", "month", "target_langs", "primary_category", "title"]].sort_values(["target_langs", "year", "id"])
rev["url"] = "https://arxiv.org/abs/" + rev.id; rev.to_csv(TAB / "cjk_lang_papers_for_review.csv", index=False)
out = {"generated": TODAY, "notebook": "analysis/notebooks/01_corpus_overview.ipynb", "period": counts["definitions"]["period"], "year_basis": YEAR_BASIS,
       "table1": {str(k): {c: (float(v) if "share" in c else int(v)) for c, v in r.items()} for k, r in t1.iterrows()},
       "table0_year_basis": {str(k): {c: int(v) for c, v in r.items()} for k, r in t0.iterrows()}, "same_title_dedup": dd,
       "sensitivity": sensitivity,
       "table1b_lang_label_review": {"per_lang": {str(k): {c: int(v) for c, v in r.items()} for k, r in per_lang.iterrows()}, "label_by_signal": {str(k): {str(c): int(v) for c, v in r.items()} for k, r in t1b.iterrows()}, "review_files": [lr["main_file"], lr["recheck_file"]]},
       "table3": {str(k): {c: float(v) for c, v in r.items()} for k, r in t3.iterrows()}}
p = VER / f"overview_notebook_{TODAY}.json"; p.write_text(json.dumps(out, ensure_ascii=False, indent=1)); print("saved", p, "| 검수 목록", len(rev), "건")